In [5]:
import pandas as pd 
import numpy as np

In [6]:
shapes = pd.read_csv("../data/gtfs_static/shapes.txt")
shapes

,shape_id,shape_pt_lat,shape_pt_lon,shape_pt_sequence,shape_dist_traveled
0,298135,49.285278,-123.115845,1,0.0000
1,298135,49.284275,-123.117371,2,0.1574
2,298135,49.283132,-123.119153,3,0.3390
3,298135,49.282477,-123.118136,4,0.4428
4,298135,49.282477,-123.118135,5,0.4429
...,...,...,...,...,...
312943,298138,49.311201,-123.082000,113,10.7423
312944,298138,49.311253,-123.082134,114,10.7536
312945,298138,49.310340,-123.083080,115,10.8762
312946,298138,49.310467,-123.083365,116,10.9013


In [7]:
shapes.head().to_dict('records')

[{'shape_id': 298135,
  'shape_pt_lat': 49.285278,
  'shape_pt_lon': -123.115845,
  'shape_pt_sequence': 1,
  'shape_dist_traveled': 0.0},
 {'shape_id': 298135,
  'shape_pt_lat': 49.284275,
  'shape_pt_lon': -123.117371,
  'shape_pt_sequence': 2,
  'shape_dist_traveled': 0.1574},
 {'shape_id': 298135,
  'shape_pt_lat': 49.283132,
  'shape_pt_lon': -123.119153,
  'shape_pt_sequence': 3,
  'shape_dist_traveled': 0.339},
 {'shape_id': 298135,
  'shape_pt_lat': 49.282477,
  'shape_pt_lon': -123.118136,
  'shape_pt_sequence': 4,
  'shape_dist_traveled': 0.4428},
 {'shape_id': 298135,
  'shape_pt_lat': 49.282477,
  'shape_pt_lon': -123.118135,
  'shape_pt_sequence': 5,
  'shape_dist_traveled': 0.4429}]

In [8]:
shapes.shape_id.nunique()

2093

In [15]:
import pandas as pd
from typing import List, Dict
import json

def shapes_to_geojson(shapes_df: pd.DataFrame) -> Dict:
    """
    Convert transit shapes data into GeoJSON format.
    
    Args:
        shapes_df: DataFrame with columns shape_id, shape_pt_lat, shape_pt_lon, 
                  shape_pt_sequence, and shape_dist_traveled
    
    Returns:
        Dictionary in GeoJSON format containing LineString features for each shape
    """
    # Sort points by shape_id and sequence to ensure correct line order
    shapes_df = shapes_df.sort_values(['shape_id', 'shape_pt_sequence'])
    
    # Initialize GeoJSON structure
    geojson = {
        "type": "FeatureCollection",
        "features": []
    }
    
    # Group points by shape_id
    for shape_id, points in shapes_df.groupby('shape_id'):
        # Create coordinates list for the LineString
        coordinates = points[['shape_pt_lon', 'shape_pt_lat']].values.tolist()
        
        # Create feature for this shape
        feature = {
            "type": "Feature",
            "properties": {
                "shape_id": shape_id,
                "total_distance": points['shape_dist_traveled'].max()
            },
            "geometry": {
                "type": "LineString",
                "coordinates": coordinates
            }
        }
        
        geojson["features"].append(feature)
    
    return geojson

In [16]:
shapes_to_geojson(shapes)

{'type': 'FeatureCollection',
 'features': [{'type': 'Feature',
   'properties': {'shape_id': 4484, 'total_distance': 0.0},
   'geometry': {'type': 'LineString',
    'coordinates': [[-122.89178, 49.224743]]}},
  {'type': 'Feature',
   'properties': {'shape_id': 292022, 'total_distance': 6.6849},
   'geometry': {'type': 'LineString',
    'coordinates': [[-123.17228, 49.257645],
     [-123.170389, 49.257617],
     [-123.168388, 49.257591],
     [-123.168389, 49.257591],
     [-123.168368, 49.257747],
     [-123.168314, 49.258663],
     [-123.168318, 49.25958],
     [-123.168293, 49.260474],
     [-123.168265, 49.261378],
     [-123.168244, 49.262254],
     [-123.168239, 49.26234],
     [-123.16822, 49.263215],
     [-123.16816, 49.26408],
     [-123.168405, 49.264979],
     [-123.168393, 49.265801],
     [-123.168359, 49.266636],
     [-123.168325, 49.267464],
     [-123.16828, 49.268381],
     [-123.168254, 49.269247],
     [-123.16822, 49.270144],
     [-123.168186, 49.271034],
     [-

In [17]:
import pandas as pd
import folium
import json
from collections import defaultdict
import numpy as np

def process_large_shape_dataset(shapes_data: list) -> dict:
    """
    Efficiently process a large dataset of transit shapes by aggregating points into distinct shapes.
    This reduces memory usage and improves visualization performance.
    
    Args:
        shapes_data: List of dictionaries containing shape points
    
    Returns:
        Dictionary containing GeoJSON FeatureCollection
    """
    # Convert to DataFrame and sort by shape_id and sequence
    df = pd.DataFrame(shapes_data).sort_values(['shape_id', 'shape_pt_sequence'])
    
    # Create a more efficient data structure using defaultdict
    shape_coordinates = defaultdict(list)
    
    # Group coordinates by shape_id without keeping all points in memory
    for _, row in df.iterrows():
        shape_coordinates[row['shape_id']].append([
            float(row['shape_pt_lon']), 
            float(row['shape_pt_lat'])
        ])
    
    # Create GeoJSON features with simplified geometries
    features = []
    for shape_id, coords in shape_coordinates.items():
        # Simplify the line by reducing points while maintaining shape
        if len(coords) > 100:  # Only simplify if we have many points
            coords = simplify_line(coords)
            
        feature = {
            "type": "Feature",
            "properties": {"shape_id": shape_id},
            "geometry": {
                "type": "LineString",
                "coordinates": coords
            }
        }
        features.append(feature)
    
    return {
        "type": "FeatureCollection",
        "features": features
    }

def simplify_line(coords: list, tolerance: float = 0.00001) -> list:
    """
    Simplify a line by removing redundant points while maintaining shape.
    Uses a simple distance-based algorithm.
    
    Args:
        coords: List of [lon, lat] coordinates
        tolerance: Distance threshold for point removal
        
    Returns:
        Simplified list of coordinates
    """
    if len(coords) <= 2:
        return coords
        
    simplified = [coords[0]]
    last_kept = coords[0]
    
    for point in coords[1:-1]:
        # Calculate distance from last kept point
        dist = np.sqrt((point[0] - last_kept[0])**2 + 
                      (point[1] - last_kept[1])**2)
        
        if dist > tolerance:
            simplified.append(point)
            last_kept = point
            
    simplified.append(coords[-1])  # Always keep the last point
    return simplified

def create_efficient_map(geojson_data: dict) -> folium.Map:
    """
    Create an interactive map optimized for large datasets.
    
    Args:
        geojson_data: GeoJSON FeatureCollection containing shapes
        
    Returns:
        folium.Map object with shapes plotted efficiently
    """
    # Calculate center from first shape's coordinates
    first_feature = geojson_data['features'][0]
    first_coords = first_feature['geometry']['coordinates']
    center_lat = first_coords[0][1]
    center_lon = first_coords[0][0]
    
    # Create base map
    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=13,
        tiles='cartodbpositron'
    )
    
    # Define color function for shapes
    def style_function(feature):
        """Generate a consistent color based on shape_id"""
        shape_id = feature['properties']['shape_id']
        # Use modulo to cycle through colors
        colors = ['#FF5733', '#33FF57', '#3357FF', '#FF33E6', '#33FFF6']
        color_index = hash(str(shape_id)) % len(colors)
        return {
            'color': colors[color_index],
            'weight': 3,
            'opacity': 0.8
        }
    
    # Add GeoJSON to map efficiently
    folium.GeoJson(
        geojson_data,
        style_function=style_function,
        name='transit_shapes'
    ).add_to(m)
    
    return m

# Example usage

# Process your large dataset
geojson_data = process_large_shape_dataset(shapes)

# Create and save the map
transit_map = create_efficient_map(geojson_data)
transit_map.save('efficient_transit_shapes.html')